import pandas as pd
import numpy as np
import sklearn
import matplotlib
import seaborn as sns
import joblib

print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Scikit-learn:", sklearn.__version__)
print("Matplotlib:", matplotlib.__version__)
print("Seaborn:", sns.__version__)
print("Joblib:", joblib.__version__)

print("Python environment is ready!")

In [8]:
%pip install pandas numpy scipy scikit-learn matplotlib seaborn joblib streamlit

Note: you may need to restart the kernel to use updated packages.


In [9]:
import pandas as pd

bank_data = pd.read_csv("bank-full.csv", sep=";")

print("Total instances:", bank_data.shape[0])
print("Total input features:", bank_data.shape[1] - 1)
print("Target column:", bank_data.columns[-1])

display(bank_data.head())

Total instances: 45211
Total input features: 16
Target column: y


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no


## 4. Dataset Inspection and Data Quality Checks

In [11]:
# Check dataset dimensions
print("Dataset Shape:", bank_data.shape)

# Check column names and data types
print("\nDataset Information:")
bank_data.info()

# Check missing values
print("\nMissing Values:")
print(bank_data.isnull().sum())

# Check duplicate records
print("\nDuplicate Records:", bank_data.duplicated().sum())

# Check target variable distribution
print("\nTarget Distribution:")
print(bank_data["y"].value_counts())

Dataset Shape: (45211, 17)

Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45211 entries, 0 to 45210
Data columns (total 17 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   age        45211 non-null  int64 
 1   job        45211 non-null  object
 2   marital    45211 non-null  object
 3   education  45211 non-null  object
 4   default    45211 non-null  object
 5   balance    45211 non-null  int64 
 6   housing    45211 non-null  object
 7   loan       45211 non-null  object
 8   contact    45211 non-null  object
 9   day        45211 non-null  int64 
 10  month      45211 non-null  object
 11  duration   45211 non-null  int64 
 12  campaign   45211 non-null  int64 
 13  pdays      45211 non-null  int64 
 14  previous   45211 non-null  int64 
 15  poutcome   45211 non-null  object
 16  y          45211 non-null  object
dtypes: int64(7), object(10)
memory usage: 5.9+ MB

Missing Values:
age          0
job          0
mar

## 5. Categorical Feature Analysis

In [13]:
# Identify categorical columns
categorical_cols = bank_data.select_dtypes(include=["object"]).columns

# Display unique values in categorical columns
for col in categorical_cols:
    print(f"\nColumn: {col}")
    print(bank_data[col].value_counts())

# Count 'unknown' values
print("\nUnknown Values:")
for col in categorical_cols:
    count = (bank_data[col] == "unknown").sum()
    print(f"{col}: {count}")


Column: job
job
blue-collar      9732
management       9458
technician       7597
admin.           5171
services         4154
retired          2264
self-employed    1579
entrepreneur     1487
unemployed       1303
housemaid        1240
student           938
unknown           288
Name: count, dtype: int64

Column: marital
marital
married     27214
single      12790
divorced     5207
Name: count, dtype: int64

Column: education
education
secondary    23202
tertiary     13301
primary       6851
unknown       1857
Name: count, dtype: int64

Column: default
default
no     44396
yes      815
Name: count, dtype: int64

Column: housing
housing
yes    25130
no     20081
Name: count, dtype: int64

Column: loan
loan
no     37967
yes     7244
Name: count, dtype: int64

Column: contact
contact
cellular     29285
unknown      13020
telephone     2906
Name: count, dtype: int64

Column: month
month
may    13766
jul     6895
aug     6247
jun     5341
nov     3970
apr     2932
feb     2649
jan     1403

## 6. Feature and Target Separation

In [15]:
# Separate input features and target variable
X = bank_data.drop(columns=["y"])
y = bank_data["y"].map({"no": 0, "yes": 1})

# Verify dimensions
print("Input features shape:", X.shape)
print("Target shape:", y.shape)

# Verify target encoding
print("\nEncoded target distribution:")
print(y.value_counts())

# Check categorical and numerical features
print("\nCategorical features:")
print(X.select_dtypes(include=["object"]).columns.tolist())

print("\nNumerical features:")
print(X.select_dtypes(include=["number"]).columns.tolist())

Input features shape: (45211, 16)
Target shape: (45211,)

Encoded target distribution:
y
0    39922
1     5289
Name: count, dtype: int64

Categorical features:
['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']

Numerical features:
['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous']


In [16]:
%pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [17]:
%pip install --upgrade --force-reinstall "numpy==2.2.6" "scipy==1.15.3" "scikit-learn==1.6.1"

  Using cached threadpoolctl-3.8.0-py3-none-any.whl.metadata (29 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 137.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.3/37.3 MB 179.9 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 138.7 MB/s eta 0:00:00
Using cached threadpoolctl-3.8.0-py3-none-any.whl (32 kB)
  Attempting uninstall: threadpoolctl
    Found existing installation: threadpoolctl 3.8.0
    Uninstalling threadpoolctl-3.8.0:
      Successfully uninstalled threadpoolctl-3.8.0
  Attempting uninstall: numpy
    Found existing installation: numpy 2.5.4
    Uninstalling numpy-2.5.4:
      Successfully uninstalled numpy-2.5.4
  Attempting uninstall: scipy
    Found existing installation: scipy 1.18.1
    Uninstalling scipy-1.18.1:
      Successfully uninstalled scipy-1.18.1
  Attempting uninstall: joblib
    Found existing installation: joblib 1.4.2
    Uninstalling joblib-1.4.2:
      Successfully uninstalled joblib-1.4

In [18]:
import numpy
import scipy
import sklearn

from sklearn.model_selection import train_test_split

print("NumPy:", numpy.__version__)
print("SciPy:", scipy.__version__)
print("Scikit-learn:", sklearn.__version__)
print("Import successful")

NumPy: 1.26.4
SciPy: 1.15.3
Scikit-learn: 1.6.1
Import successful


## 7. Train-Test Split

In [20]:
# Import train-test split function
from sklearn.model_selection import train_test_split

# Split dataset into 80% training and 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Display training and testing dimensions
print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)
print("Training target:", y_train.shape)
print("Testing target:", y_test.shape)

# Check class distribution
print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True))

print("\nTesting target distribution:")
print(y_test.value_counts(normalize=True))

Training features: (36168, 16)
Testing features: (9043, 16)
Training target: (36168,)
Testing target: (9043,)

Training target distribution:
y
0    0.883018
1    0.116982
Name: proportion, dtype: float64

Testing target distribution:
y
0    0.883003
1    0.116997
Name: proportion, dtype: float64


## 8. Data Preprocessing - Encoding and Scaling

In [22]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Identify categorical and numerical columns
categorical_features = X_train.select_dtypes(
    include=["object"]
).columns.tolist()

numerical_features = X_train.select_dtypes(
    include=["number"]
).columns.tolist()

# Create preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"),
         categorical_features)
    ]
)

print("Preprocessing pipeline created successfully")

print("Numerical features:", len(numerical_features))
print("Categorical features:", len(categorical_features))

Preprocessing pipeline created successfully
Numerical features: 7
Categorical features: 9


## 9. Logistic Regression Model

In [37]:

# Cell 9: Logistic Regression Model

from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Create Logistic Regression pipeline
lr_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

# Train model using training dataset
lr_model.fit(X_train, y_train)

# Predict using testing dataset
lr_predictions = lr_model.predict(X_test)

# Calculate model accuracy
lr_accuracy = accuracy_score(y_test, lr_predictions)

print("Logistic Regression trained successfully")
print("Logistic Regression Accuracy:", round(lr_accuracy, 4))

Logistic Regression trained successfully
Logistic Regression Accuracy: 0.9012


## 10. Decision Tree Classification Model

In [39]:
# Cell 10: Decision Tree Classification Model

from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn.base import clone

# Create Decision Tree pipeline
dt_model = Pipeline(steps=[
    ("preprocessor", clone(preprocessor)),
    ("classifier", DecisionTreeClassifier(
        random_state=42
    ))
])

# Train Decision Tree model
dt_model.fit(X_train, y_train)

# Predict using testing dataset
dt_predictions = dt_model.predict(X_test)

# Calculate model accuracy
dt_accuracy = accuracy_score(y_test, dt_predictions)

print("Decision Tree trained successfully")
print("Decision Tree Accuracy:", round(dt_accuracy, 4))


Decision Tree trained successfully
Decision Tree Accuracy: 0.8746


## 11. K-Nearest Neighbors (KNN) Classification

In [41]:
# Cell 11: K-Nearest Neighbors Classification

from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
from sklearn.base import clone

# Create KNN pipeline
knn_model = Pipeline(steps=[
    ("preprocessor", clone(preprocessor)),
    ("classifier", KNeighborsClassifier(
        n_neighbors=5
    ))
])

# Train KNN model
knn_model.fit(X_train, y_train)

# Predict using testing dataset
knn_predictions = knn_model.predict(X_test)

# Calculate model accuracy
knn_accuracy = accuracy_score(y_test, knn_predictions)

print("KNN trained successfully")
print("KNN Accuracy:", round(knn_accuracy, 4))


KNN trained successfully
KNN Accuracy: 0.8962


## 12. Naive Bayes Classification Model

In [43]:
# Cell 12: Naive Bayes Classification Model

from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import FunctionTransformer
from sklearn.base import clone
from sklearn.metrics import accuracy_score

# Create Naive Bayes pipeline
nb_model = Pipeline(steps=[
    ("preprocessor", clone(preprocessor)),
    ("to_dense", FunctionTransformer(
        lambda x: x.toarray() if hasattr(x, "toarray") else x
    )),
    ("classifier", GaussianNB())
])

# Train Naive Bayes model
nb_model.fit(X_train, y_train)

# Predict using testing dataset
nb_predictions = nb_model.predict(X_test)

# Calculate model accuracy
nb_accuracy = accuracy_score(y_test, nb_predictions)

print("Naive Bayes trained successfully")
print("Naive Bayes Accuracy:", round(nb_accuracy, 4))


Naive Bayes trained successfully
Naive Bayes Accuracy: 0.8548


## 13. Support Vector Machine (SVM) Classification

In [45]:
# Cell 13: Support Vector Machine Classification

from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.base import clone
from sklearn.metrics import accuracy_score

# Create SVM pipeline
svm_model = Pipeline(steps=[
    ("preprocessor", clone(preprocessor)),
    ("classifier", LinearSVC(
        random_state=42,
        max_iter=5000
    ))
])

# Train SVM model
svm_model.fit(X_train, y_train)

# Predict using testing dataset
svm_predictions = svm_model.predict(X_test)

# Calculate model accuracy
svm_accuracy = accuracy_score(y_test, svm_predictions)

print("SVM trained successfully")
print("SVM Accuracy:", round(svm_accuracy, 4))


SVM trained successfully
SVM Accuracy: 0.9001
